<a href="https://colab.research.google.com/github/another-lifechapter-anu20/Library-Management-System-Python/blob/main/Library_Management_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from abc import ABC, abstractmethod
from datetime import datetime, timedelta
import uuid

class LibraryItem(ABC):
    def __init__(self, title, author, item_id=None):
        self.title = title
        self.author = author
        self.item_id = item_id if item_id else str(uuid.uuid4())[:8]
        self.is_borrowed = False
        self.borrowed_by = None
        self.borrow_date = None
        self.due_date = None

    @abstractmethod
    def get_item_type(self):
        pass

    @abstractmethod
    def get_max_borrow_days(self):
        pass

    @abstractmethod
    def get_fine_per_day(self):
        pass

    def borrow(self, member_id, borrow_days=None):
        if self.is_borrowed:
            return False, "Item already borrowed"

        max_days = borrow_days if borrow_days else self.get_max_borrow_days()
        self.is_borrowed = True
        self.borrowed_by = member_id
        self.borrow_date = datetime.now()
        self.due_date = self.borrow_date + timedelta(days=max_days)
        return True, f"Borrowed until {self.due_date.strftime('%Y-%m-%d')}"

    def return_item(self):
        if not self.is_borrowed:
            return False, "Item not borrowed"

        days_overdue = 0
        fine = 0
        if datetime.now() > self.due_date:
            days_overdue = (datetime.now() - self.due_date).days
            fine = days_overdue * self.get_fine_per_day()

        self.is_borrowed = False
        self.borrowed_by = None
        self.borrow_date = None
        due_date_str = self.due_date.strftime('%Y-%m-%d') if self.due_date else "N/A"
        self.due_date = None

        return True, {
            "status": "returned",
            "due_date": due_date_str,
            "days_overdue": days_overdue,
            "fine": fine
        }

    def __str__(self):
        status = "Borrowed" if self.is_borrowed else "Available"
        return f"[{self.get_item_type()}] {self.title} by {self.author} | ID: {self.item_id} | {status}"

    def __eq__(self, other):
        return self.item_id == other.item_id

    def get_info(self):
        return {
            "id": self.item_id,
            "title": self.title,
            "author": self.author,
            "type": self.get_item_type(),
            "status": "borrowed" if self.is_borrowed else "available"
        }

In [4]:
class Book(LibraryItem):

    def __init__(self, title, author, isbn, pages, item_id=None):
        super().__init__(title, author, item_id)
        self.isbn = isbn
        self.pages = pages

    def get_item_type(self):
        return "Book"

    def get_max_borrow_days(self):
        return 14

    def get_fine_per_day(self):
        return 5

    def __str__(self):
        return f"[Book] {self.title} by {self.author} | ISBN: {self.isbn} | {self.pages} pages | ID: {self.item_id}"

class Magazine(LibraryItem):

    def __init__(self, title, publisher, issue_number, item_id=None):
        super().__init__(title, publisher, item_id)
        self.publisher = publisher
        self.issue_number = issue_number

    def get_item_type(self):
        return "Magazine"

    def get_max_borrow_days(self):
        return 7

    def get_fine_per_day(self):
        return 3

    def __str__(self):
        return f"[Magazine] {self.title} | Issue #{self.issue_number} | Publisher: {self.publisher} | ID: {self.item_id}"

class DVD(LibraryItem):

    def __init__(self, title, director, duration, item_id=None):
        super().__init__(title, director, item_id)
        self.director = director
        self.duration = duration

    def get_item_type(self):
        return "DVD"

    def get_max_borrow_days(self):
        return 5

    def get_fine_per_day(self):
        return 10

    def __str__(self):
        return f"[DVD] {self.title} directed by {self.director} | {self.duration} min | ID: {self.item_id}"

In [5]:
class Member:
    def __init__(self, name, email, phone, member_id=None):
        self.name = name
        self.email = email
        self.phone = phone
        self.member_id = member_id if member_id else str(uuid.uuid4())[:8]
        self.borrowed_items = []
        self.total_fine = 0
        self.join_date = datetime.now()

    def can_borrow(self):
        return len(self.borrowed_items) < 5 and self.total_fine == 0

    def add_borrowed_item(self, item_id):
        self.borrowed_items.append(item_id)

    def remove_borrowed_item(self, item_id):
        if item_id in self.borrowed_items:
            self.borrowed_items.remove(item_id)
            return True
        return False

    def add_fine(self, amount):
        self.total_fine += amount

    def pay_fine(self, amount):
        if amount <= self.total_fine:
            self.total_fine -= amount
            return True, f"Paid ${amount}. Remaining fine: ${self.total_fine}"
        return False, f"Amount exceeds fine. Your fine is ${self.total_fine}"

    def __str__(self):
        return f"Member: {self.name} | ID: {self.member_id} | Email: {self.email} | Borrowed: {len(self.borrowed_items)}/5 | Fine: ${self.total_fine}"

    def __eq__(self, other):
        return self.member_id == other.member_id

    def get_info(self):
        return {
            "id": self.member_id,
            "name": self.name,
            "email": self.email,
            "phone": self.phone,
            "borrowed_count": len(self.borrowed_items),
            "total_fine": self.total_fine
        }

In [6]:
class Transaction:

    def __init__(self, member_id, item_id, action):
        self.transaction_id = str(uuid.uuid4())[:8]
        self.member_id = member_id
        self.item_id = item_id
        self.action = action
        self.timestamp = datetime.now()
        self.details = {}

    def add_details(self, details):
        self.details = details

    def __str__(self):
        date_str = self.timestamp.strftime("%Y-%m-%d %H:%M")
        return f"[{date_str}] {self.action.upper()}: Member {self.member_id} -> Item {self.item_id}"

    def get_summary(self):
        return {
            "id": self.transaction_id,
            "member": self.member_id,
            "item": self.item_id,
            "action": self.action,
            "timestamp": self.timestamp.strftime("%Y-%m-%d %H:%M"),
            "details": self.details
        }

In [7]:
class Library:
    def __init__(self, name):
        self.name = name
        self.items = {}
        self.members = {}
        self.transactions = []


    def add_item(self, item):
        self.items[item.item_id] = item
        print(f"Added: {item}")
        return item.item_id

    def remove_item(self, item_id):
        if item_id in self.items:
            item = self.items[item_id]
            if item.is_borrowed:
                return False, "Cannot remove borrowed item"
            del self.items[item_id]
            return True, f"Removed: {item.title}"
        return False, "Item not found"

    def find_item(self, item_id):
        return self.items.get(item_id)

    def search_items(self, keyword):
        results = []
        keyword_lower = keyword.lower()
        for item in self.items.values():
            if keyword_lower in item.title.lower() or keyword_lower in item.author.lower():
                results.append(item)
        return results

    def add_member(self, member):
        self.members[member.member_id] = member
        print(f"Registered: {member}")
        return member.member_id

    def remove_member(self, member_id):
        if member_id in self.members:
            member = self.members[member_id]
            if member.borrowed_items:
                return False, "Member has borrowed items"
            del self.members[member_id]
            return True, f"Removed member: {member.name}"
        return False, "Member not found"

    def find_member(self, member_id):
        return self.members.get(member_id)


    def borrow_item(self, member_id, item_id, borrow_days=None):
        member = self.find_member(member_id)
        if not member:
            return False, "Member not found"

        item = self.find_item(item_id)
        if not item:
            return False, "Item not found"

        if not member.can_borrow():
            return False, f"Cannot borrow. Either already borrowed 5 items or has fine of ${member.total_fine}"

        success, message = item.borrow(member_id, borrow_days)
        if success:
            member.add_borrowed_item(item_id)
            transaction = Transaction(member_id, item_id, "borrow")
            transaction.add_details({"due_date": item.due_date.strftime("%Y-%m-%d") if item.due_date else "N/A"})
            self.transactions.append(transaction)
        return success, message

    def return_item(self, member_id, item_id):
        member = self.find_member(member_id)
        if not member:
            return False, "Member not found"

        item = self.find_item(item_id)
        if not item:
            return False, "Item not found"

        success, result = item.return_item()
        if success:
            member.remove_borrowed_item(item_id)
            if isinstance(result, dict) and result.get("fine", 0) > 0:
                member.add_fine(result["fine"])
                result["fine_added"] = result["fine"]

            transaction = Transaction(member_id, item_id, "return")
            transaction.add_details(result)
            self.transactions.append(transaction)

            if isinstance(result, dict):
                return True, result
        return success, result


    def __len__(self):
        return len(self.items)

    def __getitem__(self, key):
        if isinstance(key, str):
            return self.items.get(key)
        elif isinstance(key, int):
            return list(self.items.values())[key]
        raise TypeError("Key must be string (item_id) or integer (index)")

    def __contains__(self, item_id):
        return item_id in self.items

    def __iter__(self):
        return iter(self.items.values())

    def get_all_items(self):
        return list(self.items.values())

    def get_all_members(self):
        return list(self.members.values())

    def get_borrowed_items(self):
        return [item for item in self.items.values() if item.is_borrowed]

    def get_available_items(self):
        return [item for item in self.items.values() if not item.is_borrowed]

    def get_transaction_history(self, limit=20):
        return self.transactions[-limit:]

    def display_summary(self):
        print("=" * 50)
        print(f"{self.name} -  REPORT")
        print("=" * 50)
        print(f"Total Items: {len(self.items)}")
        print(f"  - Books: {sum(1 for i in self.items.values() if i.get_item_type() == 'Book')}")
        print(f"  - Magazines: {sum(1 for i in self.items.values() if i.get_item_type() == 'Magazine')}")
        print(f"  - DVDs: {sum(1 for i in self.items.values() if i.get_item_type() == 'DVD')}")
        print(f"  - Borrowed: {len(self.get_borrowed_items())}")
        print(f"  - Available: {len(self.get_available_items())}")
        print(f"Total Members: {len(self.members)}")
        print(f"Total Transactions: {len(self.transactions)}")
        total_fines = sum(m.total_fine for m in self.members.values())
        print(f"Total Outstanding Fines: ${total_fines}")
        print("=" * 50)

In [10]:
def create_sample_library():
    library = Library("Central Library")

    library.add_item(Book("The Great Gatsby", "F. Scott Fitzgerald", "978-0-7432-7356-5", 180))
    library.add_item(Book("1984", "George Orwell", "978-0-452-28423-4", 328))
    library.add_item(Book("To Kill a Mockingbird", "Harper Lee", "978-0-06-112008-4", 336))
    library.add_item(Magazine("National Geographic", "National Geographic Society", "2024-03"))
    library.add_item(Magazine("Time Magazine", "Time USA", "2024-12"))
    library.add_item(DVD("Inception", "Christopher Nolan", 148))
    library.add_item(DVD("The Matrix", "Wachowski Brothers", 136))

    library.add_member(Member("Aarav Sharma", "aarav@email.com", "9876543210"))
    library.add_member(Member("Vihaan Kumar", "vihaan@email.com", "9876543211"))
    library.add_member(Member("Ananya Reddy", "ananya@email.com", "9876543212"))

    return library

def run_demo():
    print("\n" + "=" * 100)
    print("LIBRARY MANAGEMENT SYSTEM - DEMO")
    print("=" * 100)

    lib = create_sample_library()

    lib.display_summary()

    print("\n SEARCH RESULTS FOR 'The':")
    results = lib.search_items("The")
    for item in results:
        print(f"  {item}")

    print("\n BORROWING ITEMS:")
    member_id = list(lib.members.keys())[0]
    item_id = list(lib.items.keys())[0]
    success, message = lib.borrow_item(member_id, item_id)
    print(f"  {message}")

    success, message = lib.borrow_item(member_id, item_id)
    print(f"  Try same item: {message}")

    print("\n RETURNING ITEM:")
    success, result = lib.return_item(member_id, item_id)
    if success and isinstance(result, dict):
        print(f"  Item returned")
        if result.get('days_overdue', 0) > 0:
            print(f"  Days overdue: {result['days_overdue']}")
            print(f"  Fine charged: ${result['fine']}")

    lib.display_summary()

run_demo()


LIBRARY MANAGEMENT SYSTEM - DEMO
Added: [Book] The Great Gatsby by F. Scott Fitzgerald | ISBN: 978-0-7432-7356-5 | 180 pages | ID: 6e7d85d5
Added: [Book] 1984 by George Orwell | ISBN: 978-0-452-28423-4 | 328 pages | ID: 0883f802
Added: [Book] To Kill a Mockingbird by Harper Lee | ISBN: 978-0-06-112008-4 | 336 pages | ID: 4c240a6e
Added: [Magazine] National Geographic | Issue #2024-03 | Publisher: National Geographic Society | ID: 2650e495
Added: [Magazine] Time Magazine | Issue #2024-12 | Publisher: Time USA | ID: 60b58733
Added: [DVD] Inception directed by Christopher Nolan | 148 min | ID: 6edb1824
Added: [DVD] The Matrix directed by Wachowski Brothers | 136 min | ID: b7f574cb
Registered: Member: Aarav Sharma | ID: 2ab4998a | Email: aarav@email.com | Borrowed: 0/5 | Fine: $0
Registered: Member: Vihaan Kumar | ID: 041b05ac | Email: vihaan@email.com | Borrowed: 0/5 | Fine: $0
Registered: Member: Ananya Reddy | ID: a96a229f | Email: ananya@email.com | Borrowed: 0/5 | Fine: $0
Central Li